In [5]:
# ── ЯЧЕЙКА 1: Установка зависимостей ─────────────────────────
!pip install -q groq

In [6]:
# ── ЯЧЕЙКА 2: Импорты и загрузка данных ──────────────────────
import pandas as pd
import numpy as np
import json
import time
import ipywidgets as widgets
from IPython.display import display, clear_output
from groq import Groq

GROQ_API_KEY = "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"

sample = pd.read_csv(
    '/content/sample_for_gemini.csv',
    sep=';',
    engine='c',
    lineterminator='\n',
    encoding='utf-8-sig',
    on_bad_lines='warn'
)

In [7]:
# ── ЯЧЕЙКА 5: LLM-разметка через Groq (zero-shot) ────────────
client = Groq(api_key=GROQ_API_KEY)

PROMPT_TEMPLATE = """Ты классификатор финансовых новостей. Для каждой статьи определи тему и выбери ОДНУ категорию:

- corp: корпоративные новости (отчётность, дивиденды, M&A, менеджмент, операционные новости компаний)
- macro: макроэкономические новости (ставка ЦБ, инфляция, ВВП, бюджет, курс рубля, но только реальные изменения, не статьи-предположения)
- geo: геополитические новости (санкции, международные отношения, ограничения, тарифные войны, даже если новость корпоративная, но есть подтекст, ее следует отнести сюда)
- irrelevant: не относится к финансовому рынку России, агрегированные новости про коммодитные рынки, дайджесты про несколько компаний

Верни ТОЛЬКО JSON вида {{"0": "corp", "1": "macro", ...}} без пояснений и без markdown.

Статьи:
{articles_text}"""

# загружаем чекпоинт если есть
try:
    checkpoint = pd.read_csv('llm_labels.csv')
    all_results = dict(zip(checkpoint['idx'].astype(str), checkpoint['llm_label']))
    print(f"Загружен чекпоинт: {len(all_results)} статей уже размечено")
except:
    all_results = {}
    print("Чекпоинт не найден, начинаем с нуля")

for batch_start in range(0, 500, 50):
    # пропускаем батчи которые уже размечены
    batch_indices = [str(i) for i in range(batch_start, batch_start+50)]
    if all(idx in all_results for idx in batch_indices):
        print(f"Батч {batch_start}-{batch_start+50} уже размечен, пропускаем")
        continue

    batch = sample.iloc[batch_start:batch_start+50]

    articles_text = ""
    for i, row in batch.iterrows():
        articles_text += f"{i}. Заголовок: {row['title']}\nТекст: {str(row['text'])[:500]}\n\n"

    batch_prompt = PROMPT_TEMPLATE.format(articles_text=articles_text)

    while True:
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": batch_prompt}],
                temperature=0,
            )
            text = response.choices[0].message.content
            batch_result = json.loads(text)
            all_results.update(batch_result)
            # сохраняем чекпоинт после каждого батча
            pd.DataFrame(list(all_results.items()), columns=['idx', 'llm_label']).to_csv('llm_labels.csv', index=False)
            print(f"Батч {batch_start}-{batch_start+50} готово, {len(all_results)} статей")
            time.sleep(5)
            break
        except Exception as e:
            print(f"Ошибка батч {batch_start}: {e}, жду 30 сек...")
            time.sleep(30)

print("Готово!")
print(pd.read_csv('llm_labels.csv')['llm_label'].value_counts())

Загружен чекпоинт: 50 статей уже размечено
Батч 0-50 уже размечен, пропускаем
Батч 50-100 готово, 100 статей
Батч 100-150 готово, 150 статей
Ошибка батч 150: Unterminated string starting at: line 1 column 12187 (char 12186), жду 30 сек...
Ошибка батч 150: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.3-70b-versatile` in organization `org_01kmtdd8dmerm93gttaw96j88e` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 12133, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}, жду 30 сек...
Ошибка батч 150: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.3-70b-versatile` in organization `org_01kmtdd8dmerm93gttaw96j88e` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 12133, please reduce your message size and try again. Need more tokens? Upgra

KeyboardInterrupt: 

In [8]:
# ── ЯЧЕЙКА 6: Сравнение ручной разметки и LLM ────────────────
labeled = pd.read_csv('labeled.csv')
llm = pd.read_csv('llm_labels.csv')

llm['idx'] = llm['idx'].astype(int)
labeled['idx'] = labeled['idx'].astype(int)

merged = labeled.merge(llm, on='idx')

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(merged['label'], merged['llm_label']))
print(confusion_matrix(merged['label'], merged['llm_label']))

              precision    recall  f1-score   support

        corp       0.72      0.82      0.77       224
         geo       0.67      0.57      0.62        49
  irrelevant       0.69      0.47      0.56       144
       macro       0.42      0.68      0.52        34

    accuracy                           0.67       451
   macro avg       0.62      0.64      0.62       451
weighted avg       0.68      0.67      0.67       451

[[184   5  22  13]
 [  9  28   8   4]
 [ 52   9  68  15]
 [ 11   0   0  23]]
